In [2]:
import polars as pl
import pandas as pd
from pathlib import Path
from datetime import date
import subprocess
DATA_DIR = Path('/workspace/data/raw')
IMAGE_DIR = Path('/workspace/data/images')
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("設定完了")

設定完了


In [3]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date, timedelta
from collections import defaultdict

DATA_DIR = Path('/workspace/data/raw')

transactions = pl.read_csv(
    DATA_DIR / 'transactions_train.csv',
    columns=['t_dat', 'customer_id', 'article_id'],
    schema_overrides={'article_id': pl.String, 'customer_id': pl.String}
).with_columns(pl.col('t_dat').str.to_date())

customers = pl.read_csv(
    DATA_DIR / 'customers.csv',
    columns=['customer_id', 'age'],
    schema_overrides={'customer_id': pl.String}
).with_columns(
    pl.when(pl.col('age').is_null()).then(-1)
    .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
    .alias('age_bin')
)

last_ts = transactions['t_dat'].max()
print(f"全データ最終日: {last_ts}, 行数: {len(transactions):,}")

全データ最終日: 2020-09-22, 行数: 31,788,324


In [3]:
transactions = pl.read_csv(
    DATA_DIR / 'transactions_train.csv',
    columns=['t_dat', 'customer_id', 'article_id'],
    schema_overrides={'article_id': pl.String, 'customer_id': pl.String}
).with_columns(pl.col('t_dat').str.to_date())

# 直近1ヶ月に購買があったユーザーをランダムに10人選ぶ
recent_users = (
    transactions
    .filter(pl.col('t_dat') >= pl.lit(date(2020, 8, 22)))
    .select('customer_id')
    .unique()
    .sample(n=10, seed=42)
    .to_series().to_list()
)

# 対象ユーザーの全期間の購買商品を取得
target_articles = (
    transactions
    .filter(pl.col('customer_id').is_in(recent_users))
    .select('article_id')
    .unique()
    .to_series().to_list()
)

print(f"対象ユーザー数: {len(recent_users)}")
print(f"対象商品数: {len(target_articles)}")
print(f"サンプルarticle_id: {target_articles[:5]}")

対象ユーザー数: 10
対象商品数: 465
サンプルarticle_id: ['0916468001', '0873884006', '0718370001', '0657215002', '0685816036']


In [4]:
# article_idから画像パスを生成
# H&MのKaggle画像パスは images/010/0108775015.jpg の形式
def get_image_path(article_id):
    return f"images/{article_id[:3]}/{article_id}.jpg"

# ダウンロード対象の画像パスリストを作成
image_paths = [get_image_path(art) for art in target_articles]
print(f"ダウンロード対象画像数: {len(image_paths)}")

# ファイルリストを作成
with open('/tmp/image_list.txt', 'w') as f:
    for path in image_paths:
        f.write(path + '\n')

print("画像リスト作成完了")

ダウンロード対象画像数: 450
画像リスト作成完了


In [ ]:
import polars as pl
import pandas as pd
from pathlib import Path
from datetime import date

DATA_DIR = Path('/workspace/data/raw')

transactions = pl.read_csv(
    DATA_DIR / 'transactions_train.csv',
    columns=['t_dat', 'customer_id', 'article_id'],
    schema_overrides={'article_id': pl.String, 'customer_id': pl.String}
).with_columns(pl.col('t_dat').str.to_date())

articles = pl.read_csv(
    DATA_DIR / 'articles.csv',
    columns=['article_id', 'prod_name', 'product_type_name', 'product_group_name', 
             'colour_group_name', 'section_name', 'garment_group_name'],
    schema_overrides={'article_id': pl.String}
)

# 直近1ヶ月に購買があったユーザーをランダムに10人選ぶ
recent_users = (
    transactions
    .filter(pl.col('t_dat') >= pl.lit(date(2020, 8, 22)))
    .select('customer_id')
    .unique()
    .sample(n=10, seed=42)
    .to_series().to_list()
)

# 対象ユーザーの全期間の購買履歴を取得
history = (
    transactions
    .filter(pl.col('customer_id').is_in(recent_users))
    .join(articles, on='article_id', how='left')
    .sort(['customer_id', 't_dat'], descending=[False, False])
)

# ユーザーごとに表示
for user in recent_users:
    user_history = history.filter(pl.col('customer_id') == user)
    print(f"\n{'='*80}")
    print(f"ユーザー: {user}")
    print(f"購買回数: {len(user_history)}件")
    print(f"{'='*80}")
    print(user_history.select([
        't_dat', 'article_id', 'prod_name', 
        'product_type_name', 'colour_group_name', 'section_name'
    ]).to_pandas().to_string(index=False))


ユーザー: be0a9b4528f4b3050cbb9282cafd1da5793c1510c9caf35021760626e50b5586
購買回数: 42件
     t_dat article_id             prod_name product_type_name colour_group_name                   section_name
2018-11-28 0556260004      Skinny L.W Elite          Trousers        Light Blue                   Ladies Denim
2018-11-28 0399087019 Super Skinny L.W Epic          Trousers             Black                   Ladies Denim
2018-11-28 0556260001      Skinny L.W Elite          Trousers              Blue                   Ladies Denim
2018-11-28 0556260001      Skinny L.W Elite          Trousers              Blue                   Ladies Denim
2018-11-28 0556260004      Skinny L.W Elite          Trousers        Light Blue                   Ladies Denim
2018-12-06 0649437010        Olga LL PJ (J)        Pyjama set        Light Grey Womens Nightwear, Socks & Tigh
2018-12-19 0653716005            Hal shorts            Shorts    Dark Turquoise                  Men H&M Sport
2018-12-19 0653716005         

In [12]:
def get_actuals(transactions, cutoff, label_days=7):
    label_end = cutoff + timedelta(days=label_days)
    grouped = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(cutoff)) & (pl.col('t_dat') <= pl.lit(label_end)))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('actual'))
    )
    return {row['customer_id']: set(row['actual']) for row in grouped.iter_rows(named=True)}

def build_global_top(transactions, cutoff, days, limit):
    start = cutoff - timedelta(days=days)
    counts = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .group_by('article_id')
        .agg(pl.len().alias('cnt'))
        .sort('cnt', descending=True)
        .head(limit)
    )
    return [(row['article_id'], row['cnt']) for row in counts.iter_rows(named=True)]

def apk(actual_set, predicted, k):
    if not actual_set:
        return 0.0
    score = 0.0
    hits = 0
    seen = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in seen:
            continue
        seen.add(article_id)
        if article_id in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)

print("関数定義完了")

関数定義完了


In [ ]:
def evaluate_repurchase_ranking(transactions, customers, cutoff, label_days=7, k_list=[12, 24, 48, 96, 120]):
    """リピート購買の2種類のランキングを比較"""
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    # 購買回数と最終購買日を集計
    purchase_stats = (
        df_train
        .group_by(['customer_id', 'article_id'])
        .agg([
            pl.len().alias('purchase_count'),
            pl.max('t_dat').alias('last_date')
        ])
    )
    
    # ①購買回数降順 → 直近購買日降順
    repurchase_dict_v1 = {}
    for row in (
        purchase_stats
        .sort(['customer_id', 'purchase_count', 'last_date'], descending=[False, True, True])
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in repurchase_dict_v1:
            repurchase_dict_v1[cid] = []
        repurchase_dict_v1[cid].append(row['article_id'])
    
    # ②直近購買日降順のみ
    repurchase_dict_v2 = {}
    for row in (
        purchase_stats
        .sort(['customer_id', 'last_date'], descending=[False, True])
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in repurchase_dict_v2:
            repurchase_dict_v2[cid] = []
        repurchase_dict_v2[cid].append(row['article_id'])
    
    # global7（補完用）
    global7 = [art for art, _ in build_global_top(transactions, cutoff, 7, max(k_list))]
    
    # 2種類を評価
    strategies = {
        '①購買回数→直近購買日': repurchase_dict_v1,
        '②直近購買日のみ': repurchase_dict_v2,
    }
    
    all_results = {}
    for name, repurchase_dict in strategies.items():
        print(f"\n--- {name} ---")
        metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
        
        for customer_id in valid_customers_list:
            actual_set = actuals[customer_id]
            candidates = list(repurchase_dict.get(customer_id, []))
            
            # global7で補完
            cand_set = set(candidates)
            fill = [art for art in global7 if art not in cand_set]
            candidates = candidates + fill
            
            for k in k_list:
                top_k = candidates[:k]
                hit = len(set(top_k) & actual_set)
                metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
                metric_acc[k]['map'].append(apk(actual_set, top_k, k))
        
        results = {}
        for k in k_list:
            results[k] = {
                'recall': np.mean(metric_acc[k]['customer_recall']),
                'map': np.mean(metric_acc[k]['map'])
            }
            print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
        
        all_results[name] = results
    
    # 比較表
    print("\n" + "="*60)
    print("比較結果")
    print("="*60)
    print(f"{'戦略':<25} {'k':>4} {'recall':>10} {'MAP':>10}")
    print("-"*55)
    for name, results in all_results.items():
        for k in k_list:
            print(f"{name:<25} {k:>4} {results[k]['recall']:>10.6f} {results[k]['map']:>10.6f}")
        print("-"*55)
    
    return all_results

evaluate_repurchase_ranking(transactions, customers, date(2020, 9, 8))

cutoff: 2020-09-08, valid_customers: 72,019


In [13]:
#半年版
def build_also_bought_dict(transactions, cutoff, threshold=10, history_days=180):
    """商品ペアの共起辞書を作成（直近半年・Polars使用）"""
    print("  商品ペア計算中...")
    
    start_date = cutoff - timedelta(days=history_days)
    df_train = transactions.filter(
        (pl.col('t_dat') > pl.lit(start_date)) &
        (pl.col('t_dat') <= pl.lit(cutoff))
    )
    
    # 同日・同ユーザーの購買ペアを作成
    also_bought = (
        df_train.select(['customer_id', 't_dat', 'article_id'])
        .join(
            df_train.select(['customer_id', 't_dat', 'article_id']),
            on=['customer_id', 't_dat'],
            how='inner'
        )
        .filter(pl.col('article_id') != pl.col('article_id_right'))
        .group_by(['article_id', 'article_id_right'])
        .agg(pl.len().alias('count'))
        .filter(pl.col('count') >= threshold)
        .sort(['article_id', 'count'], descending=[False, True])
    )
    
    also_bought_dict = defaultdict(list)
    for row in also_bought.iter_rows(named=True):
        also_bought_dict[row['article_id']].append(
            (row['article_id_right'], row['count'])
        )
    
    print(f"  共起ペア数: {len(also_bought_dict):,}商品")
    return also_bought_dict

def build_user_history_dict(transactions, cutoff, history_days=30):
    """ユーザーごとの直近1ヶ月の購買履歴を作成"""
    start_date = cutoff - timedelta(days=history_days)
    
    recent_history = (
        transactions
        .filter(
            (pl.col('t_dat') > pl.lit(start_date)) &
            (pl.col('t_dat') <= pl.lit(cutoff))
        )
        .select(['customer_id', 'article_id', 't_dat'])
        .sort(['customer_id', 't_dat'], descending=[False, True])
    )
    
    user_history_dict = defaultdict(lambda: defaultdict(list))
    for row in recent_history.iter_rows(named=True):
        user_history_dict[row['customer_id']][row['t_dat']].append(row['article_id'])
    
    return user_history_dict

def generate_also_bought_candidates(customer_id, also_bought_dict, user_history_dict, global7, k=120):
    """ユーザーごとのalso bought候補を生成"""
    history_by_date = user_history_dict.get(customer_id, {})
    if not history_by_date:
        return global7[:k]
    
    all_purchased = set(art for arts in history_by_date.values() for art in arts)
    candidates = []
    seen = set()
    
    # 直近購買日順に処理
    for purchase_date in sorted(history_by_date.keys(), reverse=True):
        articles_on_date = history_by_date[purchase_date]
        
        # 同日購買の商品を共起回数順にソート
        date_candidates = []
        for article_id in articles_on_date:
            for paired_article, count in also_bought_dict.get(article_id, []):
                if paired_article not in all_purchased and paired_article not in seen:
                    date_candidates.append((paired_article, count))
                    seen.add(paired_article)
        
        date_candidates.sort(key=lambda x: -x[1])
        candidates.extend([art for art, _ in date_candidates])
    
    # global7で補完
    for art in global7:
        if art not in seen and art not in all_purchased:
            candidates.append(art)
            seen.add(art)
    
    return candidates[:k]

def evaluate_also_bought(transactions, customers, cutoff, label_days=7,
                         threshold=10, k_list=[12, 24, 48, 96, 120]):
    """also bought候補のRecall@KとMAP@Kを計算"""
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    also_bought_dict = build_also_bought_dict(transactions, cutoff, threshold=threshold)
    user_history_dict = build_user_history_dict(transactions, cutoff)
    global7 = [art for art, _ in build_global_top(transactions, cutoff, 7, max(k_list))]
    
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
    
    for i, customer_id in enumerate(valid_customers_list):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers_list):,}")
        
        actual_set = actuals[customer_id]
        candidates = generate_also_bought_candidates(
            customer_id, also_bought_dict, user_history_dict, global7, max(k_list)
        )
        
        for k in k_list:
            top_k = candidates[:k]
            hit = len(set(top_k) & actual_set)
            metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, top_k, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

print("="*50)
print("also bought together単独評価（閾値=10, 直近半年）")
print("="*50)
also_bought_results = evaluate_also_bought(transactions, customers, date(2020, 9, 8))

also bought together単独評価（閾値=10, 直近半年）
cutoff: 2020-09-08, valid_customers: 72,019
  商品ペア計算中...


KeyboardInterrupt: 

In [14]:
def build_user_history_dict(transactions, cutoff, history_days=30):
    """ユーザーごとの直近1ヶ月の購買履歴を作成"""
    start_date = cutoff - timedelta(days=history_days)
    
    recent_history = (
        transactions
        .filter(
            (pl.col('t_dat') > pl.lit(start_date)) &
            (pl.col('t_dat') <= pl.lit(cutoff))
        )
        .select(['customer_id', 'article_id', 't_dat'])
        .sort(['customer_id', 't_dat'], descending=[False, True])
    )
    
    user_history_dict = defaultdict(lambda: defaultdict(list))
    for row in recent_history.iter_rows(named=True):
        user_history_dict[row['customer_id']][row['t_dat']].append(row['article_id'])
    
    return user_history_dict

print("関数定義完了")

関数定義完了


In [15]:
from collections import defaultdict
from datetime import date, timedelta

def get_actuals(transactions, cutoff, label_days=7):
    label_end = cutoff + timedelta(days=label_days)
    grouped = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(cutoff)) & (pl.col('t_dat') <= pl.lit(label_end)))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('actual'))
    )
    return {row['customer_id']: set(row['actual']) for row in grouped.iter_rows(named=True)}

def build_global_top(transactions, cutoff, days, limit):
    start = cutoff - timedelta(days=days)
    counts = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(start)) & (pl.col('t_dat') <= pl.lit(cutoff)))
        .group_by('article_id')
        .agg(pl.len().alias('cnt'))
        .sort('cnt', descending=True)
        .head(limit)
    )
    return [(row['article_id'], row['cnt']) for row in counts.iter_rows(named=True)]

def apk(actual_set, predicted, k):
    if not actual_set:
        return 0.0
    score = 0.0
    hits = 0
    seen = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in seen:
            continue
        seen.add(article_id)
        if article_id in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)

def build_also_bought_dict_monthly(transactions, cutoff, threshold=10):
    print("  商品ペア計算中（月ごと分割）...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    months = (
        df_train
        .select(pl.col('t_dat').dt.truncate('1mo').alias('month'))
        .unique()
        .sort('month')
        .to_series().to_list()
    )
    print(f"  対象月数: {len(months)}")
    
    all_pairs = []
    for i, month in enumerate(months):
        month_end = month + timedelta(days=31)
        df_month = df_train.filter(
            (pl.col('t_dat') >= pl.lit(month)) &
            (pl.col('t_dat') < pl.lit(month_end))
        )
        if len(df_month) == 0:
            continue
        pairs = (
            df_month.select(['customer_id', 't_dat', 'article_id'])
            .join(
                df_month.select(['customer_id', 't_dat', 'article_id']),
                on=['customer_id', 't_dat'],
                how='inner'
            )
            .filter(pl.col('article_id') != pl.col('article_id_right'))
            .group_by(['article_id', 'article_id_right'])
            .agg(pl.len().alias('count'))
        )
        all_pairs.append(pairs)
        if (i+1) % 6 == 0:
            print(f"  進捗: {i+1}/{len(months)}ヶ月完了")
    
    print("  全月分を集計中...")
    combined = (
        pl.concat(all_pairs)
        .group_by(['article_id', 'article_id_right'])
        .agg(pl.col('count').sum())
        .filter(pl.col('count') >= threshold)
        .sort(['article_id', 'count'], descending=[False, True])
    )
    
    also_bought_dict = defaultdict(list)
    for row in combined.iter_rows(named=True):
        also_bought_dict[row['article_id']].append(
            (row['article_id_right'], row['count'])
        )
    print(f"  共起ペア数: {len(also_bought_dict):,}商品")
    return also_bought_dict

def build_user_history_dict(transactions, cutoff, history_days=30):
    start_date = cutoff - timedelta(days=history_days)
    recent_history = (
        transactions
        .filter(
            (pl.col('t_dat') > pl.lit(start_date)) &
            (pl.col('t_dat') <= pl.lit(cutoff))
        )
        .select(['customer_id', 'article_id', 't_dat'])
        .sort(['customer_id', 't_dat'], descending=[False, True])
    )
    user_history_dict = defaultdict(lambda: defaultdict(list))
    for row in recent_history.iter_rows(named=True):
        user_history_dict[row['customer_id']][row['t_dat']].append(row['article_id'])
    return user_history_dict

def generate_also_bought_candidates(customer_id, also_bought_dict, user_history_dict, global7, k=120):
    history_by_date = user_history_dict.get(customer_id, {})
    if not history_by_date:
        return global7[:k]
    
    all_purchased = set(art for arts in history_by_date.values() for art in arts)
    candidates = []
    seen = set()
    
    for purchase_date in sorted(history_by_date.keys(), reverse=True):
        articles_on_date = history_by_date[purchase_date]
        date_candidates = []
        for article_id in articles_on_date:
            for paired_article, count in also_bought_dict.get(article_id, []):
                if paired_article not in all_purchased and paired_article not in seen:
                    date_candidates.append((paired_article, count))
                    seen.add(paired_article)
        date_candidates.sort(key=lambda x: -x[1])
        candidates.extend([art for art, _ in date_candidates])
    
    for art in global7:
        if art not in seen and art not in all_purchased:
            candidates.append(art)
            seen.add(art)
    
    return candidates[:k]

def evaluate_also_bought_v2(transactions, customers, cutoff, label_days=7,
                            threshold=10, k_list=[12, 24, 48, 96, 120]):
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    also_bought_dict = build_also_bought_dict_monthly(transactions, cutoff, threshold=threshold)
    user_history_dict = build_user_history_dict(transactions, cutoff)
    global7 = [art for art, _ in build_global_top(transactions, cutoff, 7, max(k_list))]
    
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
    
    for i, customer_id in enumerate(valid_customers_list):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers_list):,}")
        
        actual_set = actuals[customer_id]
        candidates = generate_also_bought_candidates(
            customer_id, also_bought_dict, user_history_dict, global7, max(k_list)
        )
        
        for k in k_list:
            top_k = candidates[:k]
            hit = len(set(top_k) & actual_set)
            metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, top_k, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

print("全関数定義完了")

全関数定義完了


In [11]:
def build_also_bought_dict_monthly(transactions, cutoff, threshold=10):
    """商品ペアの共起辞書を作成（月ごと分割・全期間）"""
    print("  商品ペア計算中（月ごと分割）...")
    
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    # 月のリストを取得
    months = (
        df_train
        .select(pl.col('t_dat').dt.truncate('1mo').alias('month'))
        .unique()
        .sort('month')
        .to_series().to_list()
    )
    print(f"  対象月数: {len(months)}")
    
    # 月ごとに計算して結合
    all_pairs = []
    for i, month in enumerate(months):
        month_end = month + timedelta(days=31)
        df_month = df_train.filter(
            (pl.col('t_dat') >= pl.lit(month)) &
            (pl.col('t_dat') < pl.lit(month_end))
        )
        
        if len(df_month) == 0:
            continue
        
        pairs = (
            df_month.select(['customer_id', 't_dat', 'article_id'])
            .join(
                df_month.select(['customer_id', 't_dat', 'article_id']),
                on=['customer_id', 't_dat'],
                how='inner'
            )
            .filter(pl.col('article_id') != pl.col('article_id_right'))
            .group_by(['article_id', 'article_id_right'])
            .agg(pl.len().alias('count'))
        )
        all_pairs.append(pairs)
        
        if (i+1) % 6 == 0:
            print(f"  進捗: {i+1}/{len(months)}ヶ月完了")
    
    # 全月分を結合して集計
    print("  全月分を集計中...")
    combined = (
        pl.concat(all_pairs)
        .group_by(['article_id', 'article_id_right'])
        .agg(pl.col('count').sum())
        .filter(pl.col('count') >= threshold)
        .sort(['article_id', 'count'], descending=[False, True])
    )
    
    also_bought_dict = defaultdict(list)
    for row in combined.iter_rows(named=True):
        also_bought_dict[row['article_id']].append(
            (row['article_id_right'], row['count'])
        )
    
    print(f"  共起ペア数: {len(also_bought_dict):,}商品")
    return also_bought_dict

# evaluate_also_bought関数のbuild_also_bought_dict呼び出しを変更
def evaluate_also_bought_v2(transactions, customers, cutoff, label_days=7,
                         threshold=10, k_list=[12, 24, 48, 96, 120]):
    """also bought候補のRecall@KとMAP@Kを計算"""
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    also_bought_dict = build_also_bought_dict_monthly(transactions, cutoff, threshold=threshold)
    user_history_dict = build_user_history_dict(transactions, cutoff)
    global7 = [art for art, _ in build_global_top(transactions, cutoff, 7, max(k_list))]
    
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
    
    for i, customer_id in enumerate(valid_customers_list):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers_list):,}")
        
        actual_set = actuals[customer_id]
        candidates = generate_also_bought_candidates(
            customer_id, also_bought_dict, user_history_dict, global7, max(k_list)
        )
        
        for k in k_list:
            top_k = candidates[:k]
            hit = len(set(top_k) & actual_set)
            metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, top_k, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

print("="*50)
print("also bought together単独評価（全期間・月分割・閾値=10）")
print("="*50)
also_bought_results = evaluate_also_bought_v2(transactions, customers, date(2020, 9, 8))

also bought together単独評価（全期間・月分割・閾値=10）
cutoff: 2020-09-08, valid_customers: 72,019
  商品ペア計算中（月ごと分割）...
  対象月数: 25
  進捗: 6/25ヶ月完了
  進捗: 12/25ヶ月完了
  進捗: 18/25ヶ月完了
  進捗: 24/25ヶ月完了
  全月分を集計中...
  共起ペア数: 51,724商品
  進捗: 0 / 72,019
  進捗: 10,000 / 72,019
  進捗: 20,000 / 72,019
  進捗: 30,000 / 72,019
  進捗: 40,000 / 72,019
  進捗: 50,000 / 72,019
  進捗: 60,000 / 72,019
  進捗: 70,000 / 72,019
  Recall@12: 0.024866  MAP@12: 0.009509
  Recall@24: 0.037213  MAP@24: 0.010268
  Recall@48: 0.059128  MAP@48: 0.010995
  Recall@96: 0.091068  MAP@96: 0.011557
  Recall@120: 0.106101  MAP@120: 0.011739


In [12]:
from gensim.models import Word2Vec
import numpy as np

def build_item2vec_model(transactions, cutoff, vector_size=64, window=5, min_count=5, epochs=10):
    """Item2Vecモデルを学習"""
    print("  購買シーケンス作成中...")
    
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    # ユーザーごとの購買シーケンスを時系列順に作成
    sequences = (
        df_train
        .sort(['customer_id', 't_dat'])
        .group_by('customer_id')
        .agg(pl.col('article_id').alias('sequence'))
    )
    
    # リストに変換
    corpus = sequences['sequence'].to_list()
    print(f"  シーケンス数: {len(corpus):,}")
    print(f"  平均シーケンス長: {np.mean([len(s) for s in corpus]):.1f}")
    
    # Word2Vec学習
    print("  Item2Vec学習中...")
    model = Word2Vec(
        sentences=corpus,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=4,
        epochs=epochs,
        sg=1  # Skip-gram
    )
    
    print(f"  学習完了: 語彙数={len(model.wv):,}")
    return model

def build_item2vec_candidates(transactions, cutoff, model, history_days=30, top_n=20):
    """Item2Vec候補生成"""
    print("  Item2Vec候補生成中...")
    
    start_date = cutoff - timedelta(days=history_days)
    recent_history = (
        transactions
        .filter(
            (pl.col('t_dat') > pl.lit(start_date)) &
            (pl.col('t_dat') <= pl.lit(cutoff))
        )
        .sort(['customer_id', 't_dat'], descending=[False, True])
        .group_by('customer_id')
        .agg(pl.col('article_id').alias('articles'))
    )
    
    user_candidates = {}
    for row in recent_history.iter_rows(named=True):
        customer_id = row['customer_id']
        purchased = row['articles']
        purchased_set = set(purchased)
        
        candidates = []
        seen = set()
        
        # 直近購買商品ごとに類似商品を取得
        for article_id in purchased:
            if article_id not in model.wv:
                continue
            similar = model.wv.most_similar(article_id, topn=top_n)
            for sim_article, score in similar:
                if sim_article not in purchased_set and sim_article not in seen:
                    candidates.append((sim_article, score))
                    seen.add(sim_article)
        
        # スコア降順でソート
        candidates.sort(key=lambda x: -x[1])
        user_candidates[customer_id] = [art for art, _ in candidates]
    
    return user_candidates

def evaluate_item2vec(transactions, customers, cutoff, label_days=7,
                      k_list=[12, 24, 48, 96, 120]):
    """Item2Vec候補のRecall@KとMAP@Kを計算"""
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    # Item2Vec学習
    model = build_item2vec_model(transactions, cutoff)
    
    # 候補生成
    user_candidates = build_item2vec_candidates(transactions, cutoff, model)
    
    # global7（補完用）
    global7 = [art for art, _ in build_global_top(transactions, cutoff, 7, max(k_list))]
    
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
    
    for i, customer_id in enumerate(valid_customers_list):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers_list):,}")
        
        actual_set = actuals[customer_id]
        candidates = user_candidates.get(customer_id, [])
        
        # global7で補完
        cand_set = set(candidates)
        fill = [art for art in global7 if art not in cand_set]
        candidates = candidates + fill
        
        for k in k_list:
            top_k = candidates[:k]
            hit = len(set(top_k) & actual_set)
            metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, top_k, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

print("="*50)
print("Item2Vec単独評価")
print("="*50)
item2vec_results = evaluate_item2vec(transactions, customers, date(2020, 9, 8))

Item2Vec単独評価
cutoff: 2020-09-08, valid_customers: 72,019
  購買シーケンス作成中...
  シーケンス数: 1,351,314
  平均シーケンス長: 23.2
  Item2Vec学習中...
  学習完了: 語彙数=90,481
  Item2Vec候補生成中...
  進捗: 0 / 72,019
  進捗: 10,000 / 72,019
  進捗: 20,000 / 72,019
  進捗: 30,000 / 72,019
  進捗: 40,000 / 72,019
  進捗: 50,000 / 72,019
  進捗: 60,000 / 72,019
  進捗: 70,000 / 72,019
  Recall@12: 0.023194  MAP@12: 0.008421
  Recall@24: 0.036158  MAP@24: 0.009198
  Recall@48: 0.060659  MAP@48: 0.010001
  Recall@96: 0.098245  MAP@96: 0.010658
  Recall@120: 0.116336  MAP@120: 0.010874


In [5]:
def evaluate_repurchase_variant_global(transactions, customers, cutoff, label_days=7,
                                        k_list=[12, 24, 48, 96, 120]):
    """repurchase → variant → global7 での候補生成評価"""
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    # 購買履歴（直近購買日順）
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    purchase_stats = (
        df_train
        .group_by(['customer_id', 'article_id'])
        .agg(pl.max('t_dat').alias('last_date'))
        .sort(['customer_id', 'last_date'], descending=[False, True])
    )
    
    repurchase_dict = {}
    for row in purchase_stats.iter_rows(named=True):
        cid = row['customer_id']
        if cid not in repurchase_dict:
            repurchase_dict[cid] = []
        repurchase_dict[cid].append(row['article_id'])
    
    # global7（補完用）
    global7 = [art for art, _ in build_global_top(transactions, cutoff, 7, max(k_list))]
    
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
    
    for i, customer_id in enumerate(valid_customers_list):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers_list):,}")
        
        actual_set = actuals[customer_id]
        
        # ① repurchase候補
        repurchase_cands = repurchase_dict.get(customer_id, [])
        cand_set = set(repurchase_cands)
        
        # ② variant候補（repurchaseの商品から派生）
        variant_cands = []
        for article_id in repurchase_cands:
            product_code = article_to_product.get(article_id)
            if product_code:
                for v in product_code_map.get(product_code, []):
                    if v != article_id and v not in cand_set:
                        variant_cands.append(v)
                        cand_set.add(v)
        
        # ③ global7で補完
        global7_fill = [art for art in global7 if art not in cand_set]
        
        candidates = repurchase_cands + variant_cands + global7_fill
        
        for k in k_list:
            top_k = candidates[:k]
            hit = len(set(top_k) & actual_set)
            metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, top_k, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

# article_to_productとproduct_code_mapが必要
articles_full = pl.read_csv(
    DATA_DIR / 'articles.csv',
    columns=['article_id', 'product_code'],
    schema_overrides={'article_id': pl.String}
)
product_code_map = {}
article_to_product = {}
for row in articles_full.iter_rows(named=True):
    code = row['product_code']
    art = row['article_id']
    article_to_product[art] = code
    if code not in product_code_map:
        product_code_map[code] = []
    product_code_map[code].append(art)

print("="*50)
print("repurchase → variant → global7 評価")
print("="*50)
results = evaluate_repurchase_variant_global(transactions, customers, date(2020, 9, 8))

repurchase → variant → global7 評価
cutoff: 2020-09-08, valid_customers: 72,019
  進捗: 0 / 72,019
  進捗: 10,000 / 72,019
  進捗: 20,000 / 72,019
  進捗: 30,000 / 72,019
  進捗: 40,000 / 72,019
  進捗: 50,000 / 72,019
  進捗: 60,000 / 72,019
  進捗: 70,000 / 72,019
  Recall@12: 0.044485  MAP@12: 0.024267
  Recall@24: 0.053211  MAP@24: 0.024816
  Recall@48: 0.068424  MAP@48: 0.025331
  Recall@96: 0.093350  MAP@96: 0.025780
  Recall@120: 0.105960  MAP@120: 0.025926


In [6]:
def evaluate_full_candidates(transactions, customers, cutoff, label_days=7,
                              n_candidates=300, k_list=[12, 24, 48, 96, 120]):
    """timedecay → repurchase → bin_popular → variant → general_pred での候補生成評価"""
    
    a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    # 時間減衰スコア
    print("  timedecay計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    
    purchase_scores = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
    )
    
    timedecay_dict = {}
    score_dict = {}
    for row in (
        purchase_scores
        .sort(['customer_id', 'score'], descending=[False, True])
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= n_candidates)
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in timedecay_dict:
            timedecay_dict[cid] = []
            score_dict[cid] = {}
        timedecay_dict[cid].append(row['article_id'])
        score_dict[cid][row['article_id']] = row['score']
    
    # repurchase（直近購買日順）
    print("  repurchase計算中...")
    repurchase_dict = {}
    for row in (
        df_train
        .group_by(['customer_id', 'article_id'])
        .agg(pl.max('t_dat').alias('last_date'))
        .sort(['customer_id', 'last_date'], descending=[False, True])
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in repurchase_dict:
            repurchase_dict[cid] = []
        repurchase_dict[cid].append(row['article_id'])
    
    # bin_popular
    print("  bin_popular計算中...")
    customers_age = customers.with_columns(
        pl.when(pl.col('age').is_null()).then(-1)
        .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
        .alias('age_bin')
    ).select(['customer_id', 'age_bin'])
    
    df_with_age = df_train.join(customers_age, on='customer_id', how='left')
    recent = df_with_age.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
    
    bin_popular_dict = {}
    for age_bin in customers_age['age_bin'].unique().to_list():
        top = (
            recent.filter(pl.col('age_bin') == age_bin)
            .group_by('article_id')
            .agg(pl.len().alias('count'))
            .sort('count', descending=True)
            .head(n_candidates)
            .select('article_id').to_series().to_list()
        )
        bin_popular_dict[age_bin] = top
    
    age_bin_dict = dict(zip(
        customers_age['customer_id'].to_list(),
        customers_age['age_bin'].to_list()
    ))
    
    # general_pred
    print("  general_pred計算中...")
    general_pred = (
        df_train.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('total_quotient'))
        .sort('total_quotient', descending=True)
        .head(n_candidates)
        .select('article_id').to_series().to_list()
    )
    
    # global7
    global7 = [art for art, _ in build_global_top(transactions, cutoff, 7, n_candidates)]
    
    # 評価
    print("  評価中...")
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
    
    for i, customer_id in enumerate(valid_customers_list):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers_list):,}")
        
        actual_set = actuals[customer_id]
        candidates = []
        cand_set = set()
        
        # ① timedecay
        for art in timedecay_dict.get(customer_id, []):
            if art not in cand_set:
                candidates.append(art)
                cand_set.add(art)
        
        # ② repurchase
        for art in repurchase_dict.get(customer_id, []):
            if art not in cand_set:
                candidates.append(art)
                cand_set.add(art)
        
        # ③ bin_popular
        age_bin = age_bin_dict.get(customer_id, -1)
        for art in bin_popular_dict.get(age_bin, []):
            if art not in cand_set:
                candidates.append(art)
                cand_set.add(art)
        
        # ④ variant
        for art in list(cand_set):
            product_code = article_to_product.get(art)
            if product_code:
                for v in product_code_map.get(product_code, []):
                    if v not in cand_set:
                        candidates.append(v)
                        cand_set.add(v)
        
        # ⑤ general_pred
        for art in general_pred:
            if art not in cand_set:
                candidates.append(art)
                cand_set.add(art)
        
        candidates = candidates[:n_candidates]
        
        for k in k_list:
            top_k = candidates[:k]
            hit = len(set(top_k) & actual_set)
            metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, top_k, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

print("="*50)
print("全戦略統合評価（300件）")
print("="*50)
full_results = evaluate_full_candidates(transactions, customers, date(2020, 9, 8))

全戦略統合評価（300件）
cutoff: 2020-09-08, valid_customers: 72,019
  timedecay計算中...
  repurchase計算中...
  bin_popular計算中...
  general_pred計算中...
  評価中...
  進捗: 0 / 72,019
  進捗: 10,000 / 72,019
  進捗: 20,000 / 72,019
  進捗: 30,000 / 72,019
  進捗: 40,000 / 72,019
  進捗: 50,000 / 72,019
  進捗: 60,000 / 72,019
  進捗: 70,000 / 72,019
  Recall@12: 0.048548  MAP@12: 0.026380
  Recall@24: 0.059238  MAP@24: 0.027010
  Recall@48: 0.077601  MAP@48: 0.027591
  Recall@96: 0.112885  MAP@96: 0.028189
  Recall@120: 0.128265  MAP@120: 0.028365


In [8]:
def evaluate_timedecay_full_v2(transactions, customers, cutoff, label_days=7, 
                                n_candidates=120, k_list=[12, 24, 48, 96, 120]):
    """時間減衰 + general_pred + bin popular + variantでRecall@Kを計算"""
    a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
    
    actuals = get_actuals(transactions, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    print(f"cutoff: {cutoff}, valid_customers: {len(valid_customers_list):,}")
    
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    
    purchase_scores = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
    )
    
    general_pred = (
        df_train.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('total_quotient'))
        .sort('total_quotient', descending=True)
        .head(n_candidates)
        .select('article_id').to_series().to_list()
    )
    
    customers_age = customers.with_columns(
        pl.when(pl.col('age').is_null()).then(-1)
        .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
        .alias('age_bin')
    ).select(['customer_id', 'age_bin'])
    
    df_with_age = df_train.join(customers_age, on='customer_id', how='left')
    recent = df_with_age.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
    
    bin_popular_dict = {}
    for age_bin in customers_age['age_bin'].unique().to_list():
        top = (
            recent.filter(pl.col('age_bin') == age_bin)
            .group_by('article_id')
            .agg(pl.len().alias('count'))
            .sort('count', descending=True)
            .head(n_candidates)
            .select('article_id').to_series().to_list()
        )
        bin_popular_dict[age_bin] = top
    
    age_bin_dict = dict(zip(
        customers_age['customer_id'].to_list(),
        customers_age['age_bin'].to_list()
    ))
    
    timedecay_dict = {}
    score_dict = {}
    for row in (
        purchase_scores
        .sort(['customer_id', 'score'], descending=[False, True])
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= n_candidates)
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in timedecay_dict:
            timedecay_dict[cid] = []
            score_dict[cid] = {}
        timedecay_dict[cid].append(row['article_id'])
        score_dict[cid][row['article_id']] = row['score']
    
    metric_acc = {k: {'customer_recall': [], 'map': []} for k in k_list}
    
    for i, customer_id in enumerate(valid_customers_list):
        if i % 10000 == 0:
            print(f"  進捗: {i:,} / {len(valid_customers_list):,}")
        
        actual_set = actuals[customer_id]
        td_cands = list(timedecay_dict.get(customer_id, []))
        cand_set = set(td_cands)
        
        # variant
        variants = []
        for art in td_cands:
            product_code = article_to_product.get(art)
            if product_code:
                for v in product_code_map.get(product_code, []):
                    if v != art and v not in cand_set:
                        variants.append(v)
                        cand_set.add(v)
        
        # bin_popular
        age_bin = age_bin_dict.get(customer_id, -1)
        bin_pop = [art for art in bin_popular_dict.get(age_bin, []) if art not in cand_set]
        cand_set.update(bin_pop)
        
        # general_pred
        gen = [art for art in general_pred if art not in cand_set]
        
        candidates = (td_cands + bin_pop + variants + gen)[:n_candidates]
        
        for k in k_list:
            top_k = candidates[:k]
            hit = len(set(top_k) & actual_set)
            metric_acc[k]['customer_recall'].append(hit / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, top_k, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['customer_recall']),
            'map': np.mean(metric_acc[k]['map'])
        }
        print(f"  Recall@{k}: {results[k]['recall']:.6f}  MAP@{k}: {results[k]['map']:.6f}")
    
    return results

print("="*50)
print("timedecay→bin→variant→general 候補数120件")
print("="*50)
results_120 = evaluate_timedecay_full_v2(transactions, customers, date(2020, 9, 8), n_candidates=120)

timedecay→bin→variant→general 候補数120件
cutoff: 2020-09-08, valid_customers: 72,019
  進捗: 0 / 72,019
  進捗: 10,000 / 72,019
  進捗: 20,000 / 72,019
  進捗: 30,000 / 72,019
  進捗: 40,000 / 72,019
  進捗: 50,000 / 72,019
  進捗: 60,000 / 72,019
  進捗: 70,000 / 72,019
  Recall@12: 0.050786  MAP@12: 0.026740
  Recall@24: 0.064896  MAP@24: 0.027582
  Recall@48: 0.087756  MAP@48: 0.028311
  Recall@96: 0.122822  MAP@96: 0.028931
  Recall@120: 0.136567  MAP@120: 0.029093


In [9]:
# transactions
print(transactions.columns)
print(transactions.head(3))

# customers
customers_full = pl.read_csv(
    DATA_DIR / 'customers.csv',
    schema_overrides={'customer_id': pl.String}
)
print(customers_full.columns)
print(customers_full.head(3))

# articles
articles_full = pl.read_csv(
    DATA_DIR / 'articles.csv',
    schema_overrides={'article_id': pl.String}
)
print(articles_full.columns)
print(articles_full.head(3))

['t_dat', 'customer_id', 'article_id']
shape: (3, 3)
┌────────────┬─────────────────────────────────┬────────────┐
│ t_dat      ┆ customer_id                     ┆ article_id │
│ ---        ┆ ---                             ┆ ---        │
│ date       ┆ str                             ┆ str        │
╞════════════╪═════════════════════════════════╪════════════╡
│ 2018-09-20 ┆ 000058a12d5b43e67d225668fa1f8d… ┆ 0663713001 │
│ 2018-09-20 ┆ 000058a12d5b43e67d225668fa1f8d… ┆ 0541518023 │
│ 2018-09-20 ┆ 00007d2de826758b65a93dd24ce629… ┆ 0505221004 │
└────────────┴─────────────────────────────────┴────────────┘
['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code']
shape: (3, 7)
┌───────────────────┬──────┬────────┬──────────────────┬──────────────────┬─────┬──────────────────┐
│ customer_id       ┆ FN   ┆ Active ┆ club_member_stat ┆ fashion_news_fre ┆ age ┆ postal_code      │
│ ---               ┆ ---  ┆ ---    ┆ us               ┆ quency        

In [7]:
# 全カラムのtransactionsを読み込み
transactions_full = pl.read_csv(
    DATA_DIR / 'transactions_train.csv',
    schema_overrides={'article_id': pl.String, 'customer_id': pl.String}
).with_columns(pl.col('t_dat').str.to_date())

# 全カラムのcustomersを読み込み
customers_full = pl.read_csv(
    DATA_DIR / 'customers.csv',
    schema_overrides={'customer_id': pl.String}
).with_columns(
    pl.when(pl.col('age').is_null()).then(-1)
    .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
    .alias('age_bin')
)

# 全カラムのarticlesを読み込み
articles_full = pl.read_csv(
    DATA_DIR / 'articles.csv',
    schema_overrides={'article_id': pl.String}
)

print(f"transactions: {len(transactions_full):,}行")
print(f"customers: {len(customers_full):,}行")
print(f"articles: {len(articles_full):,}行")
print(f"\ntransactions columns: {transactions_full.columns}")

transactions: 31,788,324行
customers: 1,371,980行
articles: 105,542行

transactions columns: ['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']


In [16]:
def create_features(transactions_full, customers_full, articles_full, cutoff, candidates_df, n_candidates=120):
    """特徴量作成"""
    print("特徴量作成中...")
    
    df_train = transactions_full.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    # ============================================================
    # Customer特徴量
    # ============================================================
    print("  Customer特徴量...")
    
    # ① age, FN, Active, club_member_status, fashion_news_frequency
    customer_attrs = customers_full.select([
        'customer_id', 'age', 'FN', 'Active', 
        'club_member_status', 'fashion_news_frequency', 'age_bin'
    ]).with_columns([
        pl.col('age').fill_null(-1),
        pl.col('FN').fill_null(0),
        pl.col('Active').fill_null(0),
        pl.col('club_member_status').fill_null('UNKNOWN'),
        pl.col('fashion_news_frequency').fill_null('NONE'),
    ])
    
    # club_member_statusをエンコード
    cms_map = {'ACTIVE': 1, 'PRE-CREATE': 0, 'LEFT CLUB': -1, 'UNKNOWN': -2}
    fnf_map = {'Regularly': 2, 'Monthly': 1, 'NONE': 0, 'None': 0}
    customer_attrs = customer_attrs.with_columns([
        pl.col('club_member_status').replace_strict(cms_map, default=-2).cast(pl.Int32).alias('club_member_status_enc'),
        pl.col('fashion_news_frequency').replace_strict(fnf_map, default=0).cast(pl.Int32).alias('fashion_news_frequency_enc'),
    ])
    
    # ② 同一アイテムの過去リピート回数（ユーザー×アイテム）
    repeat_count = (
        df_train
        .group_by(['customer_id', 'article_id'])
        .agg(pl.len().alias('repeat_count'))
    )
    
    # ③ 平均購買間隔・最終購買日からの差分
    purchase_dates = (
        df_train
        .group_by('customer_id')
        .agg([
            pl.col('t_dat').sort().alias('dates'),
            pl.col('t_dat').max().alias('last_purchase_date'),
            pl.len().alias('total_purchases')
        ])
    )
    
    # 平均購買間隔を計算
    def calc_avg_span(dates):
        if len(dates) <= 1:
            return -1.0
        diffs = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]
        return sum(diffs) / len(diffs)
    
    avg_spans = []
    for row in purchase_dates.iter_rows(named=True):
        avg_span = calc_avg_span(row['dates'])
        days_since_last = (cutoff - row['last_purchase_date']).days
        diff_from_avg = days_since_last - avg_span if avg_span > 0 else -1
        avg_spans.append({
            'customer_id': row['customer_id'],
            'avg_purchase_span': avg_span,
            'days_since_last_purchase': days_since_last,
            'diff_from_avg_span': diff_from_avg,
            'total_purchases': row['total_purchases']
        })
    
    customer_span_df = pl.DataFrame(avg_spans)
    
    # ============================================================
    # Article特徴量
    # ============================================================
    print("  Article特徴量...")
    
    # ⑥ article attributes
    article_attrs = articles_full.select([
        'article_id', 'product_code', 'product_type_no', 
        'department_no', 'section_no', 'garment_group_no', 'index_group_no'
    ])
    
    # ⑦ 週別販売数（week0〜week5）
    week_sales = []
    for w in range(6):
        start = cutoff - timedelta(days=7*(w+1))
        end = cutoff - timedelta(days=7*w)
        ws = (
            df_train
            .filter((pl.col('t_dat') > pl.lit(start)) & (pl.col('t_dat') <= pl.lit(end)))
            .group_by('article_id')
            .agg(pl.len().alias(f'week{w}_sales'))
        )
        week_sales.append(ws)
    
    article_week_sales = week_sales[0]
    for ws in week_sales[1:]:
        article_week_sales = article_week_sales.join(ws, on='article_id', how='outer_coalesce')
    
    for w in range(6):
        article_week_sales = article_week_sales.with_columns(
            pl.col(f'week{w}_sales').fill_null(0)
        )
    
    # ⑧ 日別販売数（day0〜day6）
    day_sales = []
    for day in range(7):
        target_date = cutoff - timedelta(days=day)
        ds = (
            df_train
            .filter(pl.col('t_dat') == pl.lit(target_date))
            .group_by('article_id')
            .agg(pl.len().alias(f'day{day}_sales'))
        )
        day_sales.append(ds)
    
    article_day_sales = day_sales[0]
    for ds in day_sales[1:]:
        article_day_sales = article_day_sales.join(ds, on='article_id', how='outer_coalesce')
    
    for day in range(7):
        article_day_sales = article_day_sales.with_columns(
            pl.col(f'day{day}_sales').fill_null(0)
        )
    
    # ============================================================
    # Customer×Article特徴量
    # ============================================================
    print("  Customer×Article特徴量...")
    
    # ⑨ 候補アイテムと同じproduct_typeの最終購買日
    tx_with_attrs = df_train.join(
        articles_full.select(['article_id', 'product_type_no', 'department_no', 'section_no']),
        on='article_id', how='left'
    )
    
    last_purchase_by_product_type = (
        tx_with_attrs
        .group_by(['customer_id', 'product_type_no'])
        .agg(pl.col('t_dat').max().alias('last_purchase_product_type'))
    )
    
    last_purchase_by_department = (
        tx_with_attrs
        .group_by(['customer_id', 'department_no'])
        .agg(pl.col('t_dat').max().alias('last_purchase_department'))
    )
    
    # ⑩ 同年齢ユーザーの候補アイテム購買率
    age_bin_counts = (
        df_train
        .join(customers_full.select(['customer_id', 'age_bin']), on='customer_id', how='left')
        .group_by(['age_bin', 'article_id'])
        .agg(pl.len().alias('age_bin_article_count'))
    )
    
    age_bin_totals = (
        df_train
        .join(customers_full.select(['customer_id', 'age_bin']), on='customer_id', how='left')
        .group_by('age_bin')
        .agg(pl.len().alias('age_bin_total'))
    )
    
    age_bin_purchase_rate = (
        age_bin_counts
        .join(age_bin_totals, on='age_bin', how='left')
        .with_columns(
            (pl.col('age_bin_article_count') / pl.col('age_bin_total')).alias('age_bin_purchase_rate')
        )
        .select(['age_bin', 'article_id', 'age_bin_purchase_rate'])
    )
    
    # ⑪ ユーザーの過去購買における同属性アイテムの割合
    user_product_type_counts = (
        tx_with_attrs
        .group_by(['customer_id', 'product_type_no'])
        .agg(pl.len().alias('user_product_type_count'))
    )
    
    user_total_counts = (
        df_train
        .group_by('customer_id')
        .agg(pl.len().alias('user_total_count'))
    )
    
    user_product_type_ratio = (
        user_product_type_counts
        .join(user_total_counts, on='customer_id', how='left')
        .with_columns(
            (pl.col('user_product_type_count') / pl.col('user_total_count')).alias('user_product_type_ratio')
        )
        .select(['customer_id', 'product_type_no', 'user_product_type_ratio'])
    )
    
    # ============================================================
    # 候補DFに特徴量を結合
    # ============================================================
    print("  特徴量結合中...")
    
    result = candidates_df.clone()
    
    # article属性を結合
    result = result.join(article_attrs, on='article_id', how='left')
    
    # customer属性を結合
    result = result.join(
        customer_attrs.select([
            'customer_id', 'age', 'FN', 'Active',
            'club_member_status_enc', 'fashion_news_frequency_enc', 'age_bin'
        ]),
        on='customer_id', how='left'
    )
    
    # customer span特徴量を結合
    result = result.join(customer_span_df, on='customer_id', how='left')
    
    # repeat_countを結合
    result = result.join(repeat_count, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('repeat_count').fill_null(0)
    )
    
    # 週別・日別販売数を結合
    result = result.join(article_week_sales, on='article_id', how='left')
    result = result.join(article_day_sales, on='article_id', how='left')
    for w in range(6):
        result = result.with_columns(pl.col(f'week{w}_sales').fill_null(0))
    for day in range(7):
        result = result.with_columns(pl.col(f'day{day}_sales').fill_null(0))
    
    # product_type別最終購買日を結合
    result = result.join(
        last_purchase_by_product_type,
        on=['customer_id', 'product_type_no'], how='left'
    ).with_columns(
        pl.when(pl.col('last_purchase_product_type').is_not_null())
        .then((pl.lit(cutoff) - pl.col('last_purchase_product_type')).dt.total_days())
        .otherwise(-1)
        .alias('days_since_last_product_type')
    ).drop('last_purchase_product_type')
    
    # department別最終購買日を結合
    result = result.join(
        last_purchase_by_department,
        on=['customer_id', 'department_no'], how='left'
    ).with_columns(
        pl.when(pl.col('last_purchase_department').is_not_null())
        .then((pl.lit(cutoff) - pl.col('last_purchase_department')).dt.total_days())
        .otherwise(-1)
        .alias('days_since_last_department')
    ).drop('last_purchase_department')
    
    # 年齢ビン別購買率を結合
    result = result.join(
        age_bin_purchase_rate,
        on=['age_bin', 'article_id'], how='left'
    ).with_columns(
        pl.col('age_bin_purchase_rate').fill_null(0)
    )
    
    # ユーザー×product_type割合を結合
    result = result.join(
        user_product_type_ratio,
        on=['customer_id', 'product_type_no'], how='left'
    ).with_columns(
        pl.col('user_product_type_ratio').fill_null(0)
    )
    
    print(f"  特徴量作成完了: {result.shape}")
    return result

print("関数定義完了")

関数定義完了


In [17]:
def create_reranking_dataset(transactions_full, customers_full, articles_full, cutoff, 
                              label_days=7, n_candidates=120):
    """リランキング用データセット作成"""
    print(f"\ncutoff: {cutoff}")
    
    a, b, c, d = 2.5e4, 1.5e5, 2e-1, 1e3
    
    actuals = get_actuals(transactions_full, cutoff, label_days)
    valid_customers_list = list(actuals.keys())
    
    df_train = transactions_full.filter(pl.col('t_dat') <= pl.lit(cutoff))
    
    # 時間減衰スコア
    df_train_td = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train_td.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train_td = df_train_td.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train_td = (
        df_train_td
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    a / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    b * (-c * pl.col('days_ago').cast(pl.Float64)).exp() - d
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    
    purchase_scores = (
        df_train_td.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
    )
    
    general_pred = (
        df_train_td.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('total_quotient'))
        .sort('total_quotient', descending=True)
        .head(n_candidates)
        .select('article_id').to_series().to_list()
    )
    
    timedecay_dict = {}
    score_dict = {}
    for row in (
        purchase_scores
        .sort(['customer_id', 'score'], descending=[False, True])
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= n_candidates)
        .iter_rows(named=True)
    ):
        cid = row['customer_id']
        if cid not in timedecay_dict:
            timedecay_dict[cid] = []
            score_dict[cid] = {}
        timedecay_dict[cid].append(row['article_id'])
        score_dict[cid][row['article_id']] = row['score']
    
    # bin_popular
    customers_age = customers_full.select(['customer_id', 'age_bin'])
    df_with_age = df_train.join(customers_age, on='customer_id', how='left')
    recent = df_with_age.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
    
    bin_popular_dict = {}
    for age_bin in customers_age['age_bin'].unique().to_list():
        top = (
            recent.filter(pl.col('age_bin') == age_bin)
            .group_by('article_id')
            .agg(pl.len().alias('count'))
            .sort('count', descending=True)
            .head(n_candidates)
            .select('article_id').to_series().to_list()
        )
        bin_popular_dict[age_bin] = top
    
    age_bin_dict = dict(zip(
        customers_age['customer_id'].to_list(),
        customers_age['age_bin'].to_list()
    ))
    
    # 候補生成
    rows = []
    for customer_id in valid_customers_list:
        actual_set = actuals[customer_id]
        td_cands = list(timedecay_dict.get(customer_id, []))
        cand_set = set(td_cands)
        
        # variant
        variants = []
        for art in td_cands:
            product_code = article_to_product.get(art)
            if product_code:
                for v in product_code_map.get(product_code, []):
                    if v != art and v not in cand_set:
                        variants.append(v)
                        cand_set.add(v)
        
        # bin_popular
        age_bin = age_bin_dict.get(customer_id, -1)
        bin_pop = [art for art in bin_popular_dict.get(age_bin, []) if art not in cand_set]
        cand_set.update(bin_pop)
        
        # general_pred
        gen = [art for art in general_pred if art not in cand_set]
        
        candidates = (td_cands + bin_pop + variants + gen)[:n_candidates]
        
        for rank, article_id in enumerate(candidates, start=1):
            rows.append({
                'customer_id': customer_id,
                'article_id': article_id,
                'rank': rank,
                'score': score_dict.get(customer_id, {}).get(article_id, 0.0),
                'is_history': 1 if article_id in set(td_cands) else 0,
                'is_variant': 1 if article_id in set(variants) else 0,
                'label': 1 if article_id in actual_set else 0,
            })
    
    candidates_df = pl.DataFrame(rows).with_columns([
        pl.col('customer_id').cast(pl.String),
        pl.col('article_id').cast(pl.String),
    ])
    
    print(f"  候補数: {len(candidates_df):,}行")
    print(f"  正例数: {candidates_df['label'].sum():,}")
    
    # 特徴量作成
    result = create_features(
        transactions_full, customers_full, articles_full, 
        cutoff, candidates_df, n_candidates
    )
    
    return result

print("関数定義完了")

関数定義完了


In [18]:
from catboost import CatBoost, Pool

# 特徴量カラム
feature_cols = [
    # 候補生成特徴量
    'rank', 'score', 'is_history', 'is_variant',
    # Customer特徴量
    'age', 'FN', 'Active', 'club_member_status_enc', 'fashion_news_frequency_enc',
    'avg_purchase_span', 'days_since_last_purchase', 'diff_from_avg_span', 'total_purchases',
    'repeat_count',
    # Article特徴量
    'product_code', 'product_type_no', 'department_no', 'section_no',
    'garment_group_no', 'index_group_no',
    'week0_sales', 'week1_sales', 'week2_sales', 'week3_sales', 'week4_sales', 'week5_sales',
    'day0_sales', 'day1_sales', 'day2_sales', 'day3_sales', 'day4_sales', 'day5_sales', 'day6_sales',
    # Customer×Article特徴量
    'days_since_last_product_type', 'days_since_last_department',
    'age_bin_purchase_rate', 'user_product_type_ratio',
]

train_cutoffs = [
    date(2020, 9, 15),
    date(2020, 9, 8),
    date(2020, 9, 1),
    date(2020, 8, 25),
    date(2020, 8, 18),
]

# 学習データ作成
print("学習データ作成中...")
train_dfs = []
for cutoff in train_cutoffs:
    print(f"\n{'='*40}")
    print(f"cutoff: {cutoff}")
    print(f"{'='*40}")
    df = create_reranking_dataset(transactions_full, customers_full, articles_full, cutoff)
    train_dfs.append(df.to_pandas())

train_df = pd.concat(train_dfs, ignore_index=True).sort_values('customer_id')
print(f"\n学習データ合計: {len(train_df):,}行")
print(f"正例数: {train_df['label'].sum():,}")
print(f"正例率: {train_df['label'].mean():.4f}")

学習データ作成中...

cutoff: 2020-09-15

cutoff: 2020-09-15


NameError: name 'article_to_product' is not defined